In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv('ab_data.csv')
data.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [4]:
print("Rows:", len(data))
print("\nGroup sizes:")
print(data['group'].value_counts())
print("\nLanding page counts:")
print(data['landing_page'].value_counts())

Rows: 294478

Group sizes:
group
treatment    147276
control      147202
Name: count, dtype: int64

Landing page counts:
landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64


In [5]:
# In a clean experiment: control should ALWAYS see old_page, treatment ALWAYS new_page.
# Let's check how often that's violated.
mismatch = data[
    ((data['group'] == 'control') & (data['landing_page'] == 'new_page')) |
    ((data['group'] == 'treatment') & (data['landing_page'] == 'old_page'))
]
print("Mismatched rows (group doesn't match page shown):", len(mismatch))
print("That's", f"{len(mismatch)/len(data)*100:.2f}%", "of the data")

Mismatched rows (group doesn't match page shown): 3893
That's 1.32% of the data


In [6]:
print("Duplicate user_ids:", data['user_id'].duplicated().sum())
# Look at one to understand it
dupe_ids = data[data['user_id'].duplicated(keep=False)]['user_id'].unique()
print("Example duplicated user:")
print(data[data['user_id'] == dupe_ids[0]])

Duplicate user_ids: 3894
Example duplicated user:
        user_id                   timestamp      group landing_page  converted
22       767017  2017-01-12 22:58:14.991443    control     new_page          0
277989   767017  2017-01-08 01:31:31.456648  treatment     new_page          0


In [7]:
# Keep only rows where group and page align correctly
df_clean = data[
    ((data['group'] == 'control') & (data['landing_page'] == 'old_page')) |
    ((data['group'] == 'treatment') & (data['landing_page'] == 'new_page'))
].copy()

# Drop duplicate users (keep first)
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')

print("Rows before:", len(data), "| Rows after cleaning:", len(df_clean))
print("\nClean group sizes:")
print(df_clean['group'].value_counts())

Rows before: 294478 | Rows after cleaning: 290584

Clean group sizes:
group
treatment    145310
control      145274
Name: count, dtype: int64


In [9]:
from scipy.stats import chisquare

counts = df_clean['group'].value_counts().values
srm_stat, srm_p = chisquare(counts)
print(f"SRM chi-square p-value: {srm_p:.4f}")
print("Split looks healthy (p > 0.01)" if srm_p > 0.01 else "Possible SRM, investigate the split")

SRM chi-square p-value: 0.9468
Split looks healthy (p > 0.01)
